# Base Network Description

 Describing the size of the *raw network* and its composition.

## 0. Set Up

In [1]:
from pathlib import Path
import pandas as pd

from network_pluralism.access.resolved import to_level
from network_pluralism.access.utils.files import load_matrix
from network_pluralism.setup.paths import PREPROCESSED_DATA_STATUTES, COMPUTATION_PATH

In [2]:
EXPORT_PATH = Path('../../figures_and_results/base_network_description')
EXPORT_PATH.mkdir(exist_ok=True)

## 1. Description

### 1) Actors

In [3]:
m, r, c = to_level(0, 1)
courts_len = len(r)

Out Matrix of shape (8, 93645)


In [4]:
m, r, c = to_level(1, 1)
panels_len = len(r)

Out Matrix of shape (133, 93645)


In [5]:
data = dict(
     courts=courts_len, panels=panels_len, total_actors=courts_len + panels_len
)

### 2. Documents

In [6]:
m, r, c = to_level(2, 1)
court_decisions = len(r)

Out Matrix of shape (74214, 93645)


In [7]:
m, r, c = to_level(3, 1)
paragraphs = len(r)

Out Matrix of shape (3111483, 93645)


In [8]:
number_of_statutes = len(pd.read_csv(PREPROCESSED_DATA_STATUTES / 'law_names_written.csv'))
number_of_statutes

6815

In [9]:
l_counts = []
for i in range(1,5):
    print(i)
    matrix, cols, rows = load_matrix(COMPUTATION_PATH / f"statutes_level_{i}")
    print(len(cols))
    if i == 0:
        l_counts.append({'level':0,'count':matrix.shape[0]})
    l_counts.append({'level':i+1,'count':matrix.shape[1]})

1
93645
2
268552
3
389518
4
403278


In [10]:
statutes_df = pd.read_parquet(COMPUTATION_PATH / 'levels.parquet')[["sem_id", "level"]]


In [11]:
levels_df = pd.read_parquet(COMPUTATION_PATH / 'levels.parquet')
levels_df

,sem_id,level,levels_below_min,relative_min,levels_below_max,relative_max
0,euvtrg_1,1,0,1.0,0,1.0
1,euvtrg_2,1,0,1.0,0,1.0
2,euvtrg_3,1,1,0.5,1,0.5
3,euvtrg_3_1,2,0,1.0,0,1.0
4,euvtrg_3_2,2,0,1.0,0,1.0
...,...,...,...,...,...,...
515752,auslwbentschg_20_3,2,0,1.0,0,1.0
515753,auslwbentschg_20_4,2,0,1.0,0,1.0
515754,auslwbentschg_20_5,2,0,1.0,0,1.0
515755,auslwbentschg_21,1,0,1.0,0,1.0


In [12]:
statutes_df = pd.read_pickle(PREPROCESSED_DATA_STATUTES / 'text_data.pkl')[["sem_id", "level"]]
l_counts = statutes_df.groupby("level").count().reset_index().rename(
    {"sem_id": "count"}, axis=1
)

In [13]:
number_of_sections = l_counts.set_index('level').loc[0,"count"]
number_of_sub_1_sections = l_counts.set_index('level').loc[1,"count"]
number_of_sub_2_sections = l_counts.set_index('level').loc[2,"count"]
number_of_sub_3_sections = l_counts.set_index('level').loc[3,"count"]


In [14]:
data = dict(
    data,
    number_of_sub_1_sections=number_of_sub_1_sections,
    number_of_sub_2_sections=number_of_sub_2_sections,
    number_of_sub_3_sections=number_of_sub_3_sections,
    number_of_sections=number_of_sections,
    number_of_statutes=number_of_statutes,
    number_of_paragraphs=paragraphs,
    number_of_decisions=court_decisions,
    total_num_of_docs=int(
        sum(
            [
                number_of_sub_1_sections,
                number_of_sub_2_sections,
                number_of_sub_3_sections,
                number_of_sections,
                number_of_statutes,
                paragraphs,
                court_decisions,
            ]
        )
    ),
)

In [15]:
data

{'courts': 8,
 'panels': 133,
 'total_actors': 141,
 'number_of_sub_1_sections': np.int64(208616),
 'number_of_sub_2_sections': np.int64(189374),
 'number_of_sub_3_sections': np.int64(27571),
 'number_of_sections': np.int64(91758),
 'number_of_statutes': 6815,
 'number_of_paragraphs': 3111483,
 'number_of_decisions': 74214,
 'total_num_of_docs': 3709831}

In [16]:
description_df = pd.DataFrame(data.values(),index=data.keys())
description_df

,0
courts,8
panels,133
total_actors,141
number_of_sub_1_sections,208616
number_of_sub_2_sections,189374
number_of_sub_3_sections,27571
number_of_sections,91758
number_of_statutes,6815
number_of_paragraphs,3111483
number_of_decisions,74214


In [17]:
description_df.to_csv(EXPORT_PATH / 'description.csv')